# 08 - Error Analysis and Guardrails

This notebook audits the latest baseline, LoRA, and RAG + LoRA evaluation artifacts.

In [ ]:
from pathlib import Path

import pandas as pd

from src.agents import GuardrailAgent

OUTPUT_DIR = Path('../outputs') if Path.cwd().name == 'notebooks' else Path('outputs')
comparison_df = pd.read_csv(OUTPUT_DIR / 'baseline_200_comparison_table.csv')
task_df = pd.read_csv(OUTPUT_DIR / 'baseline_200_task_metrics.csv')
lora_metrics_df = pd.read_csv(OUTPUT_DIR / 'lora_test_metrics.csv')
rag_plus_df = pd.read_csv(OUTPUT_DIR / 'rag_plus_lora_200_generations.csv')
rag_plus_metrics_df = pd.read_csv(OUTPUT_DIR / 'rag_plus_lora_200_metrics.csv')

display(comparison_df)
display(lora_metrics_df)

## RAG + LoRA Evidence-Aware Guardrails

`rag_plus_lora_200_generations.csv` includes retrieved titles and chunks, so hallucination/grounding checks are stronger than LoRA-only generation analysis.

In [ ]:
guardrail = GuardrailAgent()
rows = []
for row in rag_plus_df.itertuples(index=False):
    metric = rag_plus_metrics_df[rag_plus_metrics_df['example_id'].eq(row.example_id)].iloc[0]
    docs = str(row.retrieved_chunks).split(' ||| ') if pd.notna(row.retrieved_chunks) else []
    titles = str(row.retrieved_titles).split(' | ') if pd.notna(row.retrieved_titles) else []
    result = guardrail.validate_output(row.candidate, evidence_docs=docs, evidence_titles=titles)
    messages = [finding.message for finding in result.findings]
    failure_types = []
    if len(str(row.candidate).split()) < 12:
        failure_types.append('too_short')
    if any('Repetitive wording' in message for message in messages):
        failure_types.append('repetition')
    if metric.rougeL < 0.20:
        failure_types.append('low_rougeL')
    if metric.bleu < 5.0:
        failure_types.append('low_bleu')
    if any('weak lexical overlap' in message for message in messages):
        failure_types.append('weak_evidence_overlap')
    if not failure_types:
        failure_types.append('no_major_proxy_failure')
    rows.append({
        'example_id': row.example_id,
        'strategy': 'rag_plus_lora',
        'task': row.task,
        'topic': row.topic,
        'candidate': row.candidate,
        'reference': row.reference,
        'bleu': metric.bleu,
        'rougeL': metric.rougeL,
        'guardrail_findings': ' | '.join(messages),
        'failure_types': ', '.join(failure_types),
    })

rag_analysis_df = pd.DataFrame(rows)
rag_analysis_df.head()

In [ ]:
failure_summary = (
    rag_analysis_df.assign(failure_type=rag_analysis_df['failure_types'].str.split(', '))
    .explode('failure_type')
    .groupby(['strategy', 'failure_type'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
)

finding_summary = (
    rag_analysis_df[rag_analysis_df['guardrail_findings'].ne('')]
    .assign(message=rag_analysis_df['guardrail_findings'].str.split(' | '))
    .explode('message')
    .groupby(['strategy', 'message'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
)

display(failure_summary)
display(finding_summary)

In [ ]:
worst_cases = rag_analysis_df.sort_values(['rougeL', 'bleu'], ascending=[True, True]).head(10)
best_cases = rag_analysis_df.sort_values(['rougeL', 'bleu'], ascending=[False, False]).head(10)

display(worst_cases[['example_id', 'task', 'bleu', 'rougeL', 'failure_types', 'candidate', 'reference']])
display(best_cases[['example_id', 'task', 'bleu', 'rougeL', 'candidate', 'reference']])